# 01 – Data Collection and Preparation

## Context
This notebook is the first step of a project on the effect of diversification in a portfolio.
It builds the clean price dataset used by all the following notebooks.

## Data
- **Source:** Yahoo Finance, via `yfinance`
- **Assets:** GLD, ASML.AS, TTE.PA, AI.PA, MC.PA, IWDA.AS and BTC-USD, covering gold, cryptocurrency, European equities from different sectors and a global equity ETF
- **Period:** rolling 5 years, long enough to cover several market phases
- **Prices:** adjusted closing prices
- **Base currency:** USD, using daily exchange rates from Yahoo Finance
- **Dates kept:** only the days when all assets and exchange rates have a price
- **Output:** `Data/Processed/dataset_clean.csv`

## Contents
1. Identify each asset's currency
2. Build the required exchange rate tickers
3. Download prices and exchange rates
4. Clean the dataset
5. Convert prices to the base currency
6. Export the clean dataset

## Setup

In [8]:
import yfinance as yf
from datetime import date, timedelta
from pathlib import Path

import warnings
warnings.filterwarnings("ignore", message=".*inplace keyword.*")

In [9]:
TICKERS = ["GLD","ASML.AS","TTE.PA","AI.PA","MC.PA","IWDA.AS","BTC-USD"]
BASE_CURRENCY = "USD"
N_YEARS = 5
START = date.today() - timedelta(days=N_YEARS*365)
END = date.today()
PROCESSED_DIR = Path("../Data/Processed")

## Asset currencies

In [10]:
# --- Get the currency of each asset ---
currency_by_ticker = {}
for ticker in TICKERS :
    currency_by_ticker[ticker] = yf.Ticker(ticker).fast_info['currency']

print(f"\nCurrency of each asset: {currency_by_ticker} \n")


Currency of each asset: {'GLD': 'USD', 'ASML.AS': 'EUR', 'TTE.PA': 'EUR', 'AI.PA': 'EUR', 'MC.PA': 'EUR', 'IWDA.AS': 'EUR', 'BTC-USD': 'USD'} 



## Exchange rate tickers

In [11]:
# --- Build the exchange rate tickers ---

currencies_to_convert= set(currency_by_ticker.values()) # unique currencies
currencies_to_convert.discard(BASE_CURRENCY)# remove the base currency (does nothing if it is absent)
print(currencies_to_convert)

# --- Map each currency to its exchange rate ticker ---
fx_ticker_by_currency = {}

for currency in currencies_to_convert :
    fx_ticker_by_currency[currency] = f"{currency}{BASE_CURRENCY}=X" #"EUR" → "EURUSD=X".
print(fx_ticker_by_currency)

fx_tickers = list(fx_ticker_by_currency.values())
print(f"FX tickers : {fx_tickers}")

{'EUR'}
{'EUR': 'EURUSD=X'}
FX tickers : ['EURUSD=X']


## Download and cleaning
Adjusted prices are used because they account for dividends and stock splits, which gives a more accurate measure of returns.

To allow an efficient comparison between assets, I decided to drop the rows where a missing value is detected (see the row counts printed below).
Missing values come from different market calendars: for example, Bitcoin trades 7 days a week, while stock exchanges close on weekends and public holidays. Filling these gaps would create artificial days with a 0% return and distort volatility.

In [12]:
# --- Download the full dataset ---
raw_all = yf.download(TICKERS + fx_tickers, start=START, end=END, auto_adjust=True)
raw_close = raw_all['Close']
print(raw_close.isna().sum())
print(raw_close.shape)
raw_close = raw_close.reset_index().dropna()
print(raw_close.shape)

[*********************100%***********************]  8 of 8 completed

Ticker
AI.PA       544
ASML.AS     544
BTC-USD       0
EURUSD=X    525
GLD         570
IWDA.AS     545
MC.PA       544
TTE.PA      544
dtype: int64
(1825, 8)
(1242, 9)


## Currency conversion
All prices are converted to USD so that the assets are compared in a single currency. Returns therefore include the EUR/USD effect, as they would for a USD-based investor.

In [13]:
# --- Convert the columns to the base currency ---
dataset_clean = raw_close.copy()

column_to_convert = {}
for ticker,currency in currency_by_ticker.items() :
    if currency in fx_ticker_by_currency: 
        column_to_convert[ticker] = fx_ticker_by_currency[currency]

print(f"Columns to convert: {column_to_convert}")


for ticker_to_convert,currency_change in column_to_convert.items():
    dataset_clean[ticker_to_convert] = dataset_clean[ticker_to_convert] * dataset_clean[currency_change]
            
dataset_clean.head()

dataset_clean = dataset_clean.drop(columns=fx_tickers)



dataset_clean.head()

Columns to convert: {'ASML.AS': 'EURUSD=X', 'TTE.PA': 'EURUSD=X', 'AI.PA': 'EURUSD=X', 'MC.PA': 'EURUSD=X', 'IWDA.AS': 'EURUSD=X'}


Ticker,Date,AI.PA,ASML.AS,BTC-USD,GLD,IWDA.AS,MC.PA,TTE.PA
0,2021-11-01,114.822250,766.853481,61004.406250,167.520004,87.686347,716.846271,37.886057
1,2021-11-02,115.756863,780.777989,63226.402344,167.160004,88.330822,720.981023,38.153349
2,2021-11-03,117.157657,782.297121,62970.046875,165.770004,88.154234,717.311430,37.419958
3,2021-11-04,118.877415,804.275180,61452.230469,167.649994,89.342548,730.104315,37.834751
4,2021-11-05,118.654952,813.446271,61125.675781,169.839996,89.363297,743.508334,37.777258


## Export

In [14]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
dataset_clean.to_csv(PROCESSED_DIR / "dataset_clean.csv", index=False)